<a href="https://colab.research.google.com/github/Madelavishnu/Pytorch/blob/main/Simple_Pytorch%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from torch import nn,save,load
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# Get data
train = datasets.MNIST(root = "data", download = True,train = True,transform = ToTensor())
dataset = DataLoader(train,32)

class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1,32,(3,3)),
            nn.ReLU(),
            nn.Conv2d(32,64,(3,3)),
            nn.ReLU(),
            nn.Conv2d(64,64,(3,3)),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64*(28-6)*(28-6),10)
        )
    def forward(self,x):
        return self.model(x)


#instance of NN , Loss , Optimizer

clf = ImageClassifier().to('cuda')

opt = Adam(clf.parameters(),lr = 1e-3)
loss_fn = nn.CrossEntropyLoss()

 #training flow

if __name__ == "__main__":
    for epoch in range(10):
        for batch in dataset:
            X,y = batch
            X, y = X.to('cuda'),y.to('cuda')
            yhat = clf(X)
            loss = loss_fn(yhat,y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"Epoch :{epoch} loss is {loss.item()}")


    with open('model_state.pt', 'wb') as f:
        save(clf.state_dict(),f)

Epoch :0 loss is 0.010393832810223103
Epoch :1 loss is 0.0018047895282506943
Epoch :2 loss is 0.00014437927166000009
Epoch :3 loss is 0.0021263824310153723
Epoch :4 loss is 5.765827154391445e-05
Epoch :5 loss is 1.7881373537420586e-07
Epoch :6 loss is 9.386800229549408e-06
Epoch :7 loss is 0.00012887991033494473
Epoch :8 loss is 7.3681285357452e-06
Epoch :9 loss is 1.1250263014517259e-06


In [13]:
from torch import nn,save,load
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from PIL import Image
import torch

# Get data
train = datasets.MNIST(root = "data", download = True,train = True,transform = ToTensor())
dataset = DataLoader(train,32)

class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1,32,(3,3)),
            nn.ReLU(),
            nn.Conv2d(32,64,(3,3)),
            nn.ReLU(),
            nn.Conv2d(64,64,(3,3)),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64*(28-6)*(28-6),10)
        )

    def forward(self,x):

        return self.model(x)


#instance of NN , Loss , Optimizer

clf = ImageClassifier().to('cuda')

opt = Adam(clf.parameters(),lr = 1e-3)
loss_fn = nn.CrossEntropyLoss()



from torchvision import transforms

if __name__ == "__main__":
    # 1. Load the model state
    with open('model_state.pt', 'rb') as f:
        clf.load_state_dict(load(f))

    # 2. Open and preprocess the image
    img = Image.open('/content/Screenshot (44).png')

    # Define transformations: Convert to Grayscale ('L'), resize to 28x28, then to Tensor
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((28, 28)),
        transforms.ToTensor()
    ])

    # 3. Prepare tensor and predict
    img_tensor = transform(img).unsqueeze(0).to('cuda')

    clf.eval() # Set to evaluation mode
    with torch.no_grad():
        prediction = torch.argmax(clf(img_tensor))
        print(f"Predicted Digit: {prediction.item()}")


Predicted Digit: 5
